# Optsioonid: 12 demonstratsiooni
Käivita lahtrid järjest. Hinnad on õppenäited, mitte turuandmed. Terminal-P/L ja enne tähtaega BSM on eri mudelid.


In [ ]:
"""Euroopa optsiooni Black–Scholesi õppemudel; pole turuhinna ennustus."""
from math import erf, exp, log, sqrt

def N(x):
    return (1 + erf(x / sqrt(2))) / 2

def hind(S, K, T, sigma, r=0.0, q=0.0, liik='C'):
    if S <= 0 or K <= 0 or T < 0 or sigma <= 0:
        raise ValueError('Vaja S>0, K>0, T>=0, sigma>0')
    if liik not in ('C','P'):
        raise ValueError('liik peab olema C või P')
    if T == 0:
        return max(S-K,0) if liik=='C' else max(K-S,0)
    d1=(log(S/K)+(r-q+sigma*sigma/2)*T)/(sigma*sqrt(T))
    d2=d1-sigma*sqrt(T)
    if liik=='C':
        return S*exp(-q*T)*N(d1)-K*exp(-r*T)*N(d2)
    return K*exp(-r*T)*N(-d2)-S*exp(-q*T)*N(-d1)


## Call’i P/L
Oodatav: S=110 korral +200 USD. Muuda K, preemiat ja hindade loendit.


In [ ]:
K=105;preemia=3;kordaja=100
for S in [90,100,105,110,120]:
    payoff=max(S-K,0)
    print(S, (payoff-preemia)*kordaja)


## Put’i P/L
Oodatav: S=80 korral +1200 USD. Muuda K ja preemiat; pane tähele S=0 maksimumkasumit.


In [ ]:
K=95;preemia=3;kordaja=100
for S in [0,80,95,100,120]:
    payoff=max(K-S,0)
    print(S, (payoff-preemia)*kordaja)


## Tasuvuspiirid
Oodatav: Call104;put95. Lisa kulud ühe alusvaraühiku kohta preemiale.


In [ ]:
K=100;call_preemia=4;put_preemia=5
print("Call tähtajal:",K+call_preemia)
print("Put tähtajal:",K-put_preemia)


## Covered call’i graafik
Oodatav: Kasumi lagi800 USD;S0 korral−9700. Muuda strike’i ja preemiat; vaata nii lage kui langusriski.


In [ ]:
import matplotlib.pyplot as plt
S0=100;K=105;preemia=3;M=100
hinnad=list(range(0,151))
pnl=[(S-S0+preemia-max(S-K,0))*M for S in hinnad]
plt.plot(hinnad,pnl);plt.axhline(0,color="gray")
plt.xlabel("S tähtajal (USD)");plt.ylabel("P/L 100 aktsiale + 1 short call (USD)")
plt.title("Covered call: lõpptähtaja P/L");plt.grid();plt.show()


## Cash-secured put’i graafik
Oodatav: Reserveeritud9500;max kahju9200. Muuda strike’i; sularaha olemasolu ei muuda P/L-kuju.


In [ ]:
import matplotlib.pyplot as plt
K=95;preemia=3;M=100
hinnad=list(range(0,151))
pnl=[(preemia-max(K-S,0))*M for S in hinnad]
print("Reserveeritud osturaha:",K*M)
plt.plot(hinnad,pnl);plt.axhline(0,color="gray")
plt.xlabel("S tähtajal (USD)");plt.ylabel("P/L lepingu kohta (USD)")
plt.title("Cash-secured put: lõpptähtaja P/L");plt.grid();plt.show()


## Bull call spread’i graafik
Oodatav: Max kahju400,max kasum600. Muuda K2 ja deebetit; hoia K2>K1 ning mõistlik hind.


In [ ]:
import matplotlib.pyplot as plt
K1=100;K2=110;deebet=4;M=100
hinnad=list(range(70,141))
pnl=[(max(S-K1,0)-max(S-K2,0)-deebet)*M for S in hinnad]
plt.plot(hinnad,pnl);plt.axhline(0,color="gray")
plt.xlabel("S tähtajal (USD)");plt.ylabel("P/L kombinatsioonile (USD)")
plt.title("Bull call spread: lõpptähtaja P/L");plt.grid();plt.show()


## Straddle’i graafik
Oodatav: Tasuvus88 ja112;S100 korral−1200. Muuda kogukulu; võrdle nõutavat liikumist.


In [ ]:
import matplotlib.pyplot as plt
K=100;kogukulu=12;M=100
hinnad=list(range(50,151))
pnl=[(abs(S-K)-kogukulu)*M for S in hinnad]
plt.plot(hinnad,pnl);plt.axhline(0,color="gray")
plt.xlabel("S tähtajal (USD)");plt.ylabel("P/L kombinatsioonile (USD)")
plt.title("Long straddle: lõpptähtaja P/L");plt.grid();plt.show()


## Delta-lähendus ja täpsem mudel
Oodatav: Algdelta umbes0,5114;puutuja lahkneb suure muutuse korral. Muuda T või sigma; puutuja võib kaugel anda isegi negatiivse hinna, mis näitab lähenduse piiri.


In [ ]:
import matplotlib.pyplot as plt
from math import log,sqrt
S0=100;K=100;T=30/365;sigma=0.2
v0=hind(S0,K,T,sigma)
d1=(log(S0/K)+sigma*sigma*T/2)/(sigma*sqrt(T))
delta=N(d1)
hinnad=list(range(80,121))
plt.plot(hinnad,[hind(S,K,T,sigma) for S in hinnad],label="BSM hind")
plt.plot(hinnad,[v0+delta*(S-S0) for S in hinnad],"--",label="Lokaalne delta-lähendus")
print("Algdelta:",delta)
plt.xlabel("S täna (USD)");plt.ylabel("Call’i hind ühiku kohta (USD)")
plt.title("Enne tähtaega: 30 päeva alles, IV 20%")
plt.legend();plt.grid();plt.show()


## Ajaväärtuse mudelikõver
Oodatav: 30 päeva hind ligikaudu2,2872;null päeva hind0. Muuda S või sigma; ära käsitle kõverat turuhinna ennustusena.


In [ ]:
import matplotlib.pyplot as plt
S=K=100;sigma=0.2
paevad=list(range(0,61))
hinnad=[hind(S,K,d/365,sigma) for d in paevad]
print("30 päeva hind:",hind(S,K,30/365,sigma))
plt.plot(paevad,hinnad);plt.gca().invert_xaxis()
plt.xlabel("Allesjäänud kalendripäevad");plt.ylabel("ATM call’i mudelihind (USD/ühik)")
plt.title("Fikseeritud S=K=100, IV 20%, r=q=0")
plt.grid();plt.show()


## Kolme strateegia võrdlus
Oodatav: S120:1200,600,800 USD;kapitalivajadused erinevad. Lisa üks sama riskieelarvega kogusearvutus, mitte ainult üks leping.


In [ ]:
M=100
print("S, long call, bull call spread, covered call")
for S in [0,80,95,100,105,110,120,150]:
    call=(max(S-105,0)-3)*M
    bull=(max(S-100,0)-max(S-110,0)-4)*M
    covered=(S-100+3-max(S-105,0))*M
    print(S,call,bull,covered)


## Oodatav väärtus ja tasuvus-p
Oodatav: E=−5 USD;piir93,85%. Muuda p, võitu, kahju ja kulu; see on kahe tulemuse mudel.


In [ ]:
p=0.9;voit=10;kahju=120;kulu=2
E=p*voit-(1-p)*kahju-kulu
piir=(kahju+kulu)/(voit+kahju)
print("Oodatav tulemus:",round(E,2))
print("Tasuvuse võidumäär:",round(piir*100,2),"%")


## Riskieelarve ja täisarvuline kogus
Oodatav: 3 positsiooni;kogurisk480;eelarve500. Muuda konto või ühe positsiooni riski; tulemus0 on lubatud.


In [ ]:
from math import floor
konto=25000;riskiosa=0.02;max_kahju=150;kulureserv=10
eelarve=konto*riskiosa
n=floor(eelarve/(max_kahju+kulureserv))
print("Lepinguid:",n,"; eelarve:",eelarve,"; kogurisk:",n*(max_kahju+kulureserv))
